# Practical 8: Distributed Data Processing with PySpark DataFrames

|  |  |
|---|---|
| **Name** | Aayushman Verma |
| **Roll No.** | 57 |
| **College** | B. K. Birla College, Kalyan |
| **Student ID** | 5176202 |

**Aim:** To use PySpark to read a CSV data set with an explicit schema, perform filtering, grouping and aggregation, remove duplicate records, join two data sets and calculate the average salary of every department (with the DataFrame API and with Spark SQL).

Run this cell once if PySpark is not installed (Google Colab already has Java).

In [1]:
%pip install -q pyspark

## Task 1: Start a Spark Session and Create the Input Files

In [2]:
# Name: Ayushman Rajeev Verma
print("Name: Ayushman Rajeev Verma")
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = (SparkSession.builder.appName("Practical8_Employees").master("local[2]")
         .config("spark.ui.enabled", "false").config("spark.sql.shuffle.partitions", "4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

emp_rows = """101,Aarav Mehta,IT,65000,Pune,2021
102,Isha Nair,HR,48000,Mumbai,2019
103,Kabir Singh,IT,52000,Pune,2022
104,Meera Joshi,Finance,58000,Nashik,2020
105,Nikhil Rao,Finance,72000,Mumbai,2018
106,Pooja Desai,Sales,41000,Pune,2023
107,Sameer Khan,Sales,39000,Nashik,2024
102,Isha Nair,HR,48000,Mumbai,2019
108,Tara Iyer,IT,91000,Mumbai,2017
109,Uday Patil,Legal,60000,Pune,2022
106,Pooja Desai,Sales,41000,Pune,2023
110,Vikram Sethi,Sales,47000,Mumbai,2021"""
with open("employees.csv", "w") as f:
    f.write("emp_id,name,dept,salary,city,join_year\n" + emp_rows + "\n")
with open("departments.csv", "w") as f:
    f.write("dept,head,budget\nIT,Rohan Kulkarni,5000000\nHR,Anita Rao,2000000\nFinance,Sanjay Gupta,3500000\nSales,Neha Kapoor,4000000\n")

Name: Ayushman Rajeev Verma
Spark version: 4.0.4


## Task 2: Read a CSV Data Set with an Explicit Schema

In [3]:
# Read the CSV file with an explicit schema (no guessing) and display it
print("Name: Ayushman Rajeev Verma")
schema = StructType([
    StructField("emp_id", IntegerType(), False), StructField("name", StringType(), True),
    StructField("dept", StringType(), True),     StructField("salary", DoubleType(), True),
    StructField("city", StringType(), True),     StructField("join_year", IntegerType(), True)])

emp = spark.read.csv("employees.csv", header=True, schema=schema)
dept = spark.read.csv("departments.csv", header=True, inferSchema=True)

emp.printSchema()
print("Employees:", emp.count(), "rows | Departments:", dept.count(), "rows")
emp.show()

Name: Ayushman Rajeev Verma
root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- city: string (nullable = true)
 |-- join_year: integer (nullable = true)

Employees: 12 rows | Departments: 4 rows
+------+------------+-------+-------+------+---------+
|emp_id|        name|   dept| salary|  city|join_year|
+------+------------+-------+-------+------+---------+
|   101| Aarav Mehta|     IT|65000.0|  Pune|     2021|
|   102|   Isha Nair|     HR|48000.0|Mumbai|     2019|
|   103| Kabir Singh|     IT|52000.0|  Pune|     2022|
|   104| Meera Joshi|Finance|58000.0|Nashik|     2020|
|   105|  Nikhil Rao|Finance|72000.0|Mumbai|     2018|
|   106| Pooja Desai|  Sales|41000.0|  Pune|     2023|
|   107| Sameer Khan|  Sales|39000.0|Nashik|     2024|
|   102|   Isha Nair|     HR|48000.0|Mumbai|     2019|
|   108|   Tara Iyer|     IT|91000.0|Mumbai|     2017|
|   109|  Uday Patil|  Legal|60000.0| 

## Task 3: Filtering, Grouping and Aggregation

In [4]:
# FILTERING - column expressions and SQL-style conditions
print("Name: Ayushman Rajeev Verma")
print("Employees in Pune earning more than 50000:")
emp.filter((F.col("city") == "Pune") & (F.col("salary") > 50000)).select("name", "dept", "salary", "city").show()

print("Salary between 40000 and 60000, joined after 2020:")
emp.where("salary BETWEEN 40000 AND 60000 AND join_year > 2020").select("name", "salary", "join_year").show()

# GROUPING and AGGREGATION
print("Department summary:")
(emp.groupBy("dept")
    .agg(F.count("*").alias("headcount"), F.round(F.avg("salary"), 1).alias("avg_salary"),
         F.max("salary").alias("highest"), F.sum("salary").alias("payroll"))
    .orderBy(F.desc("payroll")).show())

Name: Ayushman Rajeev Verma
Employees in Pune earning more than 50000:
+-----------+-----+-------+----+
|       name| dept| salary|city|
+-----------+-----+-------+----+
|Aarav Mehta|   IT|65000.0|Pune|
|Kabir Singh|   IT|52000.0|Pune|
| Uday Patil|Legal|60000.0|Pune|
+-----------+-----+-------+----+

Salary between 40000 and 60000, joined after 2020:
+------------+-------+---------+
|        name| salary|join_year|
+------------+-------+---------+
| Kabir Singh|52000.0|     2022|
| Pooja Desai|41000.0|     2023|
|  Uday Patil|60000.0|     2022|
| Pooja Desai|41000.0|     2023|
|Vikram Sethi|47000.0|     2021|
+------------+-------+---------+

Department summary:
+-------+---------+----------+-------+--------+
|   dept|headcount|avg_salary|highest| payroll|
+-------+---------+----------+-------+--------+
|     IT|        3|   69333.3|91000.0|208000.0|
|  Sales|        4|   42000.0|47000.0|168000.0|
|Finance|        2|   65000.0|72000.0|130000.0|
|     HR|        2|   48000.0|48000.0| 9

## Task 4: Find and Remove Duplicate Records

In [5]:
# DUPLICATE RECORDS - first find them, then remove them
print("Name: Ayushman Rajeev Verma")
dups = emp.groupBy(emp.columns).count().filter("count > 1")
print("Duplicate records found:")
dups.show()

emp_clean = emp.dropDuplicates()
print("Rows before:", emp.count(), "| after dropDuplicates():", emp_clean.count())
print("Distinct emp_id values:", emp_clean.select("emp_id").distinct().count())

Name: Ayushman Rajeev Verma
Duplicate records found:
+------+-----------+-----+-------+------+---------+-----+
|emp_id|       name| dept| salary|  city|join_year|count|
+------+-----------+-----+-------+------+---------+-----+
|   102|  Isha Nair|   HR|48000.0|Mumbai|     2019|    2|
|   106|Pooja Desai|Sales|41000.0|  Pune|     2023|    2|
+------+-----------+-----+-------+------+---------+-----+

Rows before: 12 | after dropDuplicates(): 10
Distinct emp_id values: 10


## Task 5: Join Two Data Sets

In [6]:
# JOINS
print("Name: Ayushman Rajeev Verma")
inner = emp_clean.join(dept, on="dept", how="inner")
print("Inner join - employees with their department head:")
inner.select("emp_id", "name", "dept", "head", "salary").orderBy("emp_id").show()

print("Left anti join - employees whose department is missing in departments.csv:")
emp_clean.join(dept, on="dept", how="left_anti").select("emp_id", "name", "dept").show()

print("Department budget per employee:")
(inner.groupBy("dept", "budget").agg(F.count("*").alias("staff"))
      .withColumn("budget_per_employee", F.round(F.col("budget") / F.col("staff"))).orderBy("dept").show())

Name: Ayushman Rajeev Verma
Inner join - employees with their department head:
+------+------------+-------+--------------+-------+
|emp_id|        name|   dept|          head| salary|
+------+------------+-------+--------------+-------+
|   101| Aarav Mehta|     IT|Rohan Kulkarni|65000.0|
|   102|   Isha Nair|     HR|     Anita Rao|48000.0|
|   103| Kabir Singh|     IT|Rohan Kulkarni|52000.0|
|   104| Meera Joshi|Finance|  Sanjay Gupta|58000.0|
|   105|  Nikhil Rao|Finance|  Sanjay Gupta|72000.0|
|   106| Pooja Desai|  Sales|   Neha Kapoor|41000.0|
|   107| Sameer Khan|  Sales|   Neha Kapoor|39000.0|
|   108|   Tara Iyer|     IT|Rohan Kulkarni|91000.0|
|   110|Vikram Sethi|  Sales|   Neha Kapoor|47000.0|
+------+------------+-------+--------------+-------+

Left anti join - employees whose department is missing in departments.csv:
+------+----------+-----+
|emp_id|      name| dept|
+------+----------+-----+
|   109|Uday Patil|Legal|
+------+----------+-----+

Department budget per emp

## Task 6: Average Salary by Department (DataFrame API and Spark SQL)

In [7]:
# AVERAGE SALARY BY DEPARTMENT - DataFrame API
print("Name: Ayushman Rajeev Verma")
avg_df = emp_clean.groupBy("dept").agg(F.round(F.avg("salary"), 2).alias("average_salary")).orderBy(F.desc("average_salary"))
avg_df.show()

# the same result with Spark SQL
emp_clean.createOrReplaceTempView("employees")
spark.sql("SELECT dept, ROUND(AVG(salary), 2) AS average_salary FROM employees GROUP BY dept ORDER BY average_salary DESC").show()

# extra: highest paid employee in every department (window function)
w = Window.partitionBy("dept").orderBy(F.desc("salary"))
emp_clean.withColumn("rank_in_dept", F.rank().over(w)).filter("rank_in_dept = 1").select("dept", "name", "salary").orderBy("dept").show()
spark.stop()

Name: Ayushman Rajeev Verma
+-------+--------------+
|   dept|average_salary|
+-------+--------------+
|     IT|      69333.33|
|Finance|       65000.0|
|  Legal|       60000.0|
|     HR|       48000.0|
|  Sales|      42333.33|
+-------+--------------+

+-------+--------------+
|   dept|average_salary|
+-------+--------------+
|     IT|      69333.33|
|Finance|       65000.0|
|  Legal|       60000.0|
|     HR|       48000.0|
|  Sales|      42333.33|
+-------+--------------+

+-------+------------+-------+
|   dept|        name| salary|
+-------+------------+-------+
|Finance|  Nikhil Rao|72000.0|
|     HR|   Isha Nair|48000.0|
|     IT|   Tara Iyer|91000.0|
|  Legal|  Uday Patil|60000.0|
|  Sales|Vikram Sethi|47000.0|
+-------+------------+-------+



**Conclusion:** PySpark was used to load employee data with a defined schema, filter and aggregate it, locate and remove duplicate rows, join it with department details (including an anti-join to find unmatched departments) and calculate the average salary per department using both the DataFrame API and Spark SQL.